# 06 보완판: 금리와 가격 (구×월 패널)
기존 06의 금리 파트(급변 1건 비교)를 월별 금리 변화 전체를 쓰는 패널 회귀로 대체한다. 금리 자료가 2024-08-20까지라 분석 구간은 2024-07까지다.

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display
root = next(p for p in [Path.cwd(), Path.cwd().parent] if (p/'data').is_dir())
policy_dir = next((root/'data').glob('06*'))
out = root/'outputs'/'06_interest_panel'
out.mkdir(parents=True, exist_ok=True)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
def save(frame, name):
    frame.to_csv(out/(name+'.csv'), index=False, encoding='utf-8-sig')

## 1. 실거래 정제 (기존 06과 동일 기준)

In [2]:
raw = pd.read_excel(root/'data'/'서울_아파트_매매_실거래가_2017-2026_통합.xlsx', dtype=str)
raw.columns = raw.columns.str.strip()
for c in raw:
    raw[c] = raw[c].astype('string').str.strip()
def num(s): return pd.to_numeric(s.str.replace(',', '', regex=False), errors='coerce')
df = pd.DataFrame({
    '가격': num(raw['거래금액(만원)']), '면적': num(raw['전용면적(㎡)']),
    '층': num(raw['층']), '건축년도': num(raw['건축년도']),
    '날짜': pd.to_datetime(raw['계약년월']+raw['계약일'].str.zfill(2), format='%Y%m%d', errors='coerce'),
    '구': raw['시군구'].str.extract(r'^서울특별시\s+(\S+구)', expand=False),
    '유형': raw['주택유형'], '해제': raw['해제사유발생일'].fillna('')})
ok = df['구'].notna() & df['유형'].eq('아파트') & df['날짜'].notna() & df['가격'].gt(0) & df['면적'].gt(0) & df['해제'].isin(['', '-', '0'])
df = df[ok.fillna(False)].drop(columns=['유형', '해제']).copy()
df['월'] = df['날짜'].dt.to_period('M').dt.to_timestamp()
df = df[df['월'] >= '2017-11-01'].copy()   # 첫 달(2017-10)은 부분 집계
df['로그가격'] = np.log(df['가격']/df['면적'])
print(f'{len(df):,}건')

529,030건


## 2. 구성보정 가격지수
구×월 고정효과에 면적·연식·층 구간을 통제해(FWL) 구×월 로그가격 수준을 구한다. 거래수 10건 미만인 구-월은 제외.

In [3]:
age = (df['월'].dt.year - df['건축년도']).clip(lower=0, upper=60)
ctrl = pd.concat([
    pd.get_dummies(pd.cut(df['면적'], [0, 40, 60, 85, 102, 135, 1e4]).astype(str), prefix='면적', drop_first=True),
    pd.get_dummies(pd.cut(age, [-1, 5, 10, 15, 20, 25, 30, 100]).astype(str), prefix='연식', drop_first=True),
    pd.get_dummies(pd.cut(df['층'], [-100, 2, 5, 10, 15, 20, 200]).astype(str), prefix='층', drop_first=True),
], axis=1).astype(float)
key = df['구'] + '|' + df['월'].dt.strftime('%Y-%m')
y = df['로그가격']
beta = np.linalg.lstsq((ctrl - ctrl.groupby(key).transform('mean')).to_numpy(),
                       (y - y.groupby(key).transform('mean')).to_numpy(), rcond=None)[0]
adj = y - ctrl.to_numpy() @ beta
cell = adj.groupby([df['구'], df['월']]).agg(['mean', 'size'])
cell.columns = ['로그지수', '거래수']
cell.loc[cell['거래수'] < 10, '로그지수'] = np.nan
idx = cell['로그지수'].unstack(0)
idx = idx.reindex(pd.date_range(idx.index.min(), idx.index.max(), freq='MS'))
print('제외된 구-월:', int(idx.isna().sum().sum()), '| 구 평균 보정지수와 서울 중위가격 월변화 상관:',
      round(idx.diff().mean(axis=1).corr(df.groupby('월')['로그가격'].median().diff()), 3))

제외된 구-월: 16 | 구 평균 보정지수와 서울 중위가격 월변화 상관: 0.653


## 3. 금리
CD수익률, 국고채3년의 월평균과 월간 변화(%p). 완결 월까지만 사용.

In [4]:
rate_raw = pd.read_csv(next(policy_dir.glob('*시장금리*.csv')), encoding='cp949')
rate_raw['날짜'] = pd.to_datetime(rate_raw['기준일자'])
rate_raw['금리'] = pd.to_numeric(rate_raw['고시금리'], errors='coerce')
last_full = rate_raw['날짜'].max().to_period('M').to_timestamp() - pd.DateOffset(months=1)
rate_lv = pd.DataFrame({
    label: rate_raw[rate_raw['시장금리종류'].eq(name)].groupby('날짜')['금리'].mean().resample('MS').mean()
    for name, label in [('CD수익률', 'CD'), ('국고채3년', '국고채3Y')]})
rate_d = rate_lv[rate_lv.index <= last_full].diff()
idx = idx[idx.index <= last_full]
print('분석 종료월:', f'{last_full:%Y-%m}', '| |Δ CD|≥0.25%p인 달:', int((rate_d['CD'].reindex(idx.index).abs() >= .25).sum()))
display(rate_d.reindex(idx.index).describe().T[['count', 'mean', 'std', 'min', 'max']])

분석 종료월: 2024-07 | |Δ CD|≥0.25%p인 달: 5


,count,mean,std,min,max
CD,81.000,0.027,0.140,-0.282,0.679
국고채3Y,81.000,0.013,0.177,-0.424,0.656


## 4. 모형
종속변수는 월간 지수 변화율(%, 서울은 구 동일가중 평균). 설명변수는 Δ금리 0~L개월 시차, 통제는 전월·전전월 상승률, 정책월 더미(발생월+다음 달, 방향 미구분), 구 고정효과. 패널 표준오차는 Driscoll–Kraay, 서울 시계열은 HAC.

In [5]:
events = pd.read_csv(root/'outputs'/'06_policy'/'event_calendar.csv', parse_dates=['월'])
ev = pd.DatetimeIndex(events['월'])
policy_dummy = pd.Series(idx.index.isin(ev) | idx.index.isin(ev + pd.DateOffset(months=1)), index=idx.index).astype(float)
dY = idx.diff()*100
dY_seoul = dY.mean(axis=1)   # 서울 = 구 동일가중 평균 변화율
series_gu = {g: dY[g] for g in dY.columns}

def build(series_map, rate, L, leads=0, policy=True, start=None, end=None):
    r = rate_d[rate].reindex(pd.date_range(rate_d.index.min(), idx.index.max(), freq='MS'))
    X = pd.DataFrame({f'dr_{k}': r.shift(k).reindex(idx.index) for k in range(-leads, L+1)})
    cols = list(X.columns)
    frames = []
    for name, s in series_map.items():
        d = X.copy()
        d['y'] = s
        d['mom_1'], d['mom_2'] = s.shift(1), s.shift(2)
        if policy: d['policy'] = policy_dummy
        d['구'], d['t'] = name, np.arange(len(d))
        frames.append(d)
    p = pd.concat(frames).rename_axis('월').reset_index().dropna()
    if start: p = p[p['월'] >= start]
    if end: p = p[p['월'] <= end]
    return p, cols

class Fit:
    pass

def fit(p, cols, fe=True, maxlags=6):
    # OLS + 시차 가중(Bartlett) 공분산. 구 고정효과가 있으면 같은 달의 구들을 합산하는 Driscoll-Kraay, 없으면 Newey-West.
    X = p[cols + [c for c in p.columns if c.startswith('mom_') or c == 'policy']].copy()
    if fe: X = pd.concat([X, pd.get_dummies(p['구'], drop_first=True).astype(float)], axis=1)
    X.insert(0, 'const', 1.0)
    A, yv = X.to_numpy(float), p['y'].to_numpy(float)
    XtXi = np.linalg.pinv(A.T @ A)
    beta = XtXi @ A.T @ yv
    u = yv - A @ beta
    h = pd.DataFrame(A*u[:, None]).groupby(p['t'].to_numpy()).sum()
    h = h.reindex(range(int(h.index.min()), int(h.index.max())+1), fill_value=0.0).to_numpy()
    S = h.T @ h
    for l in range(1, maxlags+1):
        G = h[l:].T @ h[:-l]
        S += (1 - l/(maxlags+1)) * (G + G.T)
    r = Fit()
    r.names, r.params, r.V, r.nobs = list(X.columns), beta, XtXi @ S @ XtXi, len(yv)
    r.df = h.shape[0] - 1
    return r

def lincomb(res, cols):
    R = np.zeros((len(cols), len(res.names)))
    for i, c in enumerate(cols): R[i, res.names.index(c)] = 1
    return R

def cum(res, cols):
    # 금리 +1%p 변화 시 가격 변화율(%p): 시차 계수 합, 표준오차, p값
    r = lincomb(res, cols).sum(axis=0)
    e, s = float(r @ res.params), float(np.sqrt(r @ res.V @ r))
    return e, s, float(2*stats.t.sf(abs(e/s), res.df))

def coef_table(res, cols):
    i = [res.names.index(c) for c in cols]
    b, s = res.params[i], np.sqrt(np.diag(res.V)[i])
    return pd.DataFrame({'계수': b, '표준오차': s, 'p값': 2*stats.t.sf(abs(b/s), res.df)}, index=cols)

def wald_p(res, cols):
    R = lincomb(res, cols)
    b = R @ res.params
    return float(stats.chi2.sf(b @ np.linalg.pinv(R @ res.V @ R.T) @ b, len(cols)))

### 서울 전체 (CD, 0~6개월)

In [6]:
ps, cs = build({'서울': dY_seoul}, 'CD', 6)
res_s = fit(ps, cs, fe=False)
display(coef_table(res_s, cs))
est_s, se_s, p_s = cum(res_s, cs)
print(f'누적 {est_s:+.2f}%p (SE {se_s:.2f}, p={p_s:.3f}), {res_s.nobs}개월')

,계수,표준오차,p값
dr_0,-7.248,2.341,0.003
dr_1,-0.372,2.336,0.874
dr_2,-2.243,2.029,0.272
dr_3,1.379,1.827,0.453
dr_4,-4.254,1.821,0.022
dr_5,1.551,2.420,0.524
dr_6,0.843,1.356,0.536


누적 -10.34%p (SE 3.28, p=0.002), 78개월


### 구×월 패널 (CD, 0~6개월)

In [7]:
pp, cp = build(series_gu, 'CD', 6)
res_p = fit(pp, cp)
tab = coef_table(res_p, cp)
display(tab)
est_p, se_p, p_p = cum(res_p, cp)
print(f'누적 {est_p:+.2f}%p (SE {se_p:.2f}, p={p_p:.3f}), {res_p.nobs:,}개 구-월')
save(tab.rename_axis('변수').reset_index(), 'panel_lag_coefficients')

,계수,표준오차,p값
dr_0,-9.511,2.137,0.000
dr_1,-3.212,2.273,0.162
dr_2,-1.459,1.836,0.429
dr_3,0.010,2.063,0.996
dr_4,-4.930,1.772,0.007
dr_5,-0.196,2.123,0.927
dr_6,2.024,1.643,0.222


누적 -17.27%p (SE 4.17, p=0.000), 1,910개 구-월


## 5. 플라시보
미래 금리(t+1~t+3)를 추가해 계수가 모두 0인지 검정한다. 유의하면 금리→가격이 아니라 공통 추세·선행 관계 가능성이 있다.

In [8]:
pl, cl = build(series_gu, 'CD', 6, leads=3)
res_l = fit(pl, cl)
lead = [c for c in cl if c.split('_')[1].startswith('-')]
placebo_p = wald_p(res_l, lead)
display(coef_table(res_l, lead))
print(f'동시 0 검정 p={placebo_p:.3f}')

,계수,표준오차,p값
dr_-3,0.516,3.070,0.867
dr_-2,-0.969,2.201,0.661
dr_-1,-2.081,1.898,0.277


동시 0 검정 p=0.621


## 6. 강건성
금리 종류, 시차 길이, 정책더미 유무, 하위기간을 바꿔 누적 반응이 유지되는지 확인.

In [9]:
rows = []
for rate in ['CD', '국고채3Y']:
    for L in [3, 6, 12]:
        for pol in [True, False]:
            for span, (a, b) in {'전체': (None, None), '~2021': (None, '2021-12-01'), '2022~': ('2022-01-01', None)}.items():
                try:
                    pn, cr = build(series_gu, rate, L, policy=pol, start=a, end=b)
                    e, s, p = cum(fit(pn, cr, maxlags=min(L, 6)), cr)
                except Exception:
                    e = s = p = np.nan
                rows.append({'금리': rate, '시차': f'0~{L}', '정책더미': pol, '기간': span, '누적반응': e, '표준오차': s, 'p값': p})
robust = pd.DataFrame(rows)
save(robust, 'robustness')
display(robust)
full = robust[robust['기간'].eq('전체')].dropna(subset=['누적반응'])
agree = (np.sign(full['누적반응']) == np.sign(est_p)).mean()
sig_share = (full['p값'] < .05).mean()
print(f'전체기간 {len(full)}개 규격: 부호 일치 {agree:.0%}, p<0.05 {sig_share:.0%}')

,금리,시차,정책더미,기간,누적반응,표준오차,p값
0,CD,0~3,True,전체,-16.452,3.983,0.000
1,CD,0~3,True,~2021,-25.309,6.532,0.000
2,CD,0~3,True,2022~,-10.842,2.918,0.001
3,CD,0~3,False,전체,-17.124,3.641,0.000
4,CD,0~3,False,~2021,-23.206,5.800,0.000
5,CD,0~3,False,2022~,-14.805,3.561,0.000
6,CD,0~6,True,전체,-17.274,4.167,0.000
7,CD,0~6,True,~2021,-21.686,9.436,0.026
8,CD,0~6,True,2022~,-12.599,3.609,0.002
9,CD,0~6,False,전체,-17.767,3.830,0.000


전체기간 12개 규격: 부호 일치 100%, p<0.05 100%


## 7. 구별 민감도
구마다 같은 모형(CD, 0~6개월)을 따로 추정하고 가격대와 비교한다.

In [10]:
rows = []
for g in dY.columns:
    pn, cr = build({g: dY[g]}, 'CD', 6)
    if len(pn) >= 40:
        e, s, p = cum(fit(pn, cr, fe=False), cr)
        rows.append({'자치구': g, '누적반응': e, '표준오차': s, 'p값': p})
by_gu = pd.DataFrame(rows).merge(np.exp(idx.mean()).rename('평균가격수준'), left_on='자치구', right_index=True)
by_gu['가격대'] = pd.qcut(by_gu['평균가격수준'], 3, labels=['저가', '중가', '고가'])
by_gu = by_gu.sort_values('누적반응')
save(by_gu, 'by_district_sensitivity')
display(by_gu)
rho, rho_p = stats.spearmanr(by_gu['평균가격수준'], by_gu['누적반응'])
print(f'가격수준-민감도 Spearman ρ={rho:.2f} (p={rho_p:.3f}) | 가격대별 평균 누적반응:')
display(by_gu.groupby('가격대', observed=True)['누적반응'].mean())

,자치구,누적반응,표준오차,p값,평균가격수준,가격대
22,종로구,-49.480,18.259,0.008,"1,081.318",중가
23,중구,-40.630,12.020,0.001,"1,122.653",중가
2,강북구,-37.913,7.819,0.000,791.277,저가
20,용산구,-33.050,13.321,0.015,"1,746.438",고가
5,광진구,-30.614,6.531,0.000,"1,311.279",고가
7,금천구,-26.392,5.311,0.000,744.167,저가
11,동작구,-24.001,6.836,0.001,"1,252.578",고가
24,중랑구,-23.630,3.220,0.000,826.016,저가
15,성동구,-22.920,4.321,0.000,"1,469.036",고가
16,성북구,-21.712,3.429,0.000,927.647,저가


가격수준-민감도 Spearman ρ=0.18 (p=0.383) | 가격대별 평균 누적반응:


가격대
저가   -22.067
중가   -19.146
고가   -18.902
Name: 누적반응, dtype: float64

## 8. 판정

In [11]:
checks = pd.Series({
    '서울 시계열 누적효과 p<0.05': p_s < .05,
    '구 패널 누적효과 p<0.05': p_p < .05,
    '미래 금리 플라시보 통과(p≥0.05)': placebo_p >= .05,
    '강건성 부호 일치 ≥80%': agree >= .8,
    '강건성 p<0.05 비율 ≥50%': sig_share >= .5,
}, name='충족')
save(checks.rename_axis('기준').reset_index(), 'verdict_checks')
display(checks)
print(f'{int(checks.sum())}/{len(checks)} 충족 | 구 패널 누적효과 {est_p:+.2f}%p / 금리 +1%p')

서울 시계열 누적효과 p<0.05       True
구 패널 누적효과 p<0.05         True
미래 금리 플라시보 통과(p≥0.05)    True
강건성 부호 일치 ≥80%           True
강건성 p<0.05 비율 ≥50%       True
Name: 충족, dtype: bool

5/5 충족 | 구 패널 누적효과 -17.27%p / 금리 +1%p


## 한계
- 분석 구간(2017-11~2024-07)에 금리 인상 국면은 2022년 1회뿐이라, 그 시기의 다른 충격과 분리되지 않는다.
- 금리는 전 구 공통 충격이라 시간고정효과를 쓸 수 없고, 공통 추세와 겹칠 수 있다.
- 정책 더미는 방향을 구분하지 않은 통제용이다.
- 2024-08 이후 금리가 없어 2025~2026년 상승기는 설명하지 못한다.